In [1]:
import pandas as pd
import os
import re

In [25]:
def combine_snapshots(pokemon_name, folder="../pokemon_snapshots"):
    dfs = []

    # Normalize name for matching
    safe_name = pokemon_name.replace(" ", "_").lower()

    for filename in os.listdir(folder):
        if filename.endswith(".csv") and filename.startswith(safe_name):
            filepath = os.path.join(folder, filename)
            df = pd.read_csv(filepath)

            # Extract date from filename: psyduck_pricing_2026-07-10.csv
            match = re.search(r"(\d{4}-\d{2}-\d{2})", filename)
            snapshot_date = match.group(1) if match else None
            df["snapshot_date"] = snapshot_date

            dfs.append(df)

    if not dfs:
        raise ValueError(f"No CSV snapshots found for Pokémon: {pokemon_name}")

    combined = pd.concat(dfs, ignore_index=True)

    # Sort by card + date
    combined = combined.sort_values(["id", "snapshot_date"]).reset_index(drop=True)

    # Remove exact duplicates
    combined = combined.drop_duplicates()

    return combined

In [26]:

def preprocess_combined_df(df, pokemon_name, output_folder="../data"):
    # 1. Remove columns that are entirely NaN
    df = df.dropna(axis=1, how="all")

    # 2. Remove columns that are entirely empty strings
    empty_cols = [
        col for col in df.columns
        if df[col].astype(str).str.strip().eq("").all()
    ]
    df = df.drop(columns=empty_cols)

    # 3. Identify pricing columns
    pricing_cols = [col for col in df.columns if col.startswith(("cm_", "tp_"))]

    # 4. Remove rows with no pricing at all
    df = df.dropna(subset=pricing_cols, how="all").reset_index(drop=True)

    # --- SAVE CLEANED DATAFRAME ---
    os.makedirs(output_folder, exist_ok=True)

    safe_name = pokemon_name.replace(" ", "_").lower()
    output_path = os.path.join(output_folder, f"{safe_name}_combined_clean.csv")

    df.to_csv(output_path, index=False)

    return df, output_path


In [27]:
pokemon_list = ["psyduck", "pikachu", "kadabra"]

for pokemon in pokemon_list:

    combined_df = combine_snapshots(pokemon)
    clean_df, output_path = preprocess_combined_df(combined_df, pokemon)

    print(f"Saved cleaned combined file for {pokemon} to: {output_path}\n")


Saved cleaned combined file for psyduck to: ../data/psyduck_combined_clean.csv

Saved cleaned combined file for pikachu to: ../data/pikachu_combined_clean.csv

Saved cleaned combined file for kadabra to: ../data/kadabra_combined_clean.csv



In [ ]:
combined_set = combine_snapshots('chaos_rising', folder='../pricing_snapshots')



In [24]:
combined_set

,id,name,localId,rarity,supertype,cm_updated,cm_unit,cm_idProduct,cm_avg,cm_low,...,tp_normal_highPrice,tp_normal_marketPrice,tp_normal_directLowPrice,tp_holofoil_productId,tp_holofoil_lowPrice,tp_holofoil_midPrice,tp_holofoil_highPrice,tp_holofoil_marketPrice,tp_holofoil_directLowPrice,snapshot_date
0,me04-001,Weedle,1,Common,NaN,2026-07-06T18:05:01.396Z,EUR,886393,0.05,0.02,...,5.00,0.09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-06
1,me04-001,Weedle,1,Common,NaN,2026-07-07T18:05:09.149Z,EUR,886393,0.05,0.02,...,5.00,0.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-07
2,me04-001,Weedle,1,Common,NaN,2026-07-08T18:05:11.087Z,EUR,886393,0.05,0.02,...,5.00,0.09,0.05,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-08
3,me04-001,Weedle,1,Common,NaN,2026-07-09T18:05:09.065Z,EUR,886393,0.05,0.02,...,5.00,0.18,0.05,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-09
4,me04-001,Weedle,1,Common,NaN,2026-07-10T18:05:08.528Z,EUR,886393,0.05,0.02,...,10.01,0.19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
971,me04-122,Mega Greninja ex,122,Mega Hyper Rare,NaN,2026-07-09T18:05:09.065Z,EUR,886515,183.99,130.00,...,NaN,NaN,NaN,693518.0,198.49,245.40,1000.0,208.88,219.0,2026-07-09
972,me04-122,Mega Greninja ex,122,Mega Hyper Rare,NaN,2026-07-10T18:05:08.528Z,EUR,886515,183.99,128.00,...,NaN,NaN,NaN,693518.0,198.01,243.67,4321.0,205.66,NaN,2026-07-10
973,me04-122,Mega Greninja ex,122,Mega Hyper Rare,NaN,2026-07-11T18:05:08.252Z,EUR,886515,183.99,128.00,...,NaN,NaN,NaN,693518.0,182.90,236.84,4321.0,184.71,NaN,2026-07-11
974,me04-122,Mega Greninja ex,122,Mega Hyper Rare,NaN,2026-07-12T18:05:09.094Z,EUR,886515,183.99,128.00,...,NaN,NaN,NaN,693518.0,176.30,237.83,4321.0,179.80,NaN,2026-07-12
